# 5주차 · 00 데이터 준비 — 수업 데이터를 Google Drive에 저장하기

수업 시간에 GitHub 접속이 불안정하거나, **미리 받아 둔 데이터로만 수업**하고 싶을 때 쓰는 노트북입니다.

| 누가 | 언제 | 무엇을 |
|---|---|---|
| **강의자** | 수업 전 1회 (약 1분) | 이 노트북을 실행 → Drive `강의5주/02_데이터`에 실습 CSV 6개 + 확인용 목록 + ZIP 저장 |
| 학생 | 수업 중 | 공유받은 `02_데이터` 폴더를 **내 드라이브에 바로가기 추가** → 실습 노트북에서 `DATA_SOURCE = "drive"` |

- 저장되는 파일은 GitHub 수업 원본과 **MD5까지 같은 파일**이므로 모든 기대값이 그대로 나옵니다.
- 실행하면 Google Drive 접근 허용 창이 뜹니다 → 본인 계정 선택 → 허용.

In [ ]:
import os, io, glob, hashlib, zipfile, urllib.request
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
GITHUB_BASE = "https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week05/data/"
MANIFEST = "week05_MANIFEST.csv"
EXTRA = ["DATA_SOURCES.md"]

## 1. 저장할 폴더 정하기
내 드라이브에서 `강의5주/02_데이터` 폴더를 자동으로 찾습니다. 못 찾거나 다른 곳에 저장하려면 `TARGET`에 경로를 직접 적으세요.
(Drive 화면의 폴더 경로가 `내 드라이브 > A > B > 강의5주 > 02_데이터` 이면 `/content/drive/MyDrive/A/B/강의5주/02_데이터`)

In [ ]:
TARGET = ""          # 비워 두면 자동 탐색. 예: "/content/drive/MyDrive/동서울대학교/__인공지능과데이터활용_목야34/강의5주/02_데이터"

if not TARGET:
    for depth in range(0, 7):
        hits = sorted(glob.glob("/content/drive/MyDrive/" + "*/" * depth + "강의5주/02_데이터"))
        if hits:
            TARGET = hits[0]
            break
if not TARGET:
    TARGET = "/content/drive/MyDrive/Week05_data"     # 못 찾으면 새 폴더에 저장
os.makedirs(TARGET, exist_ok=True)
print("저장 위치:", TARGET)

## 2. GitHub 원본 내려받기 → MD5 확인 → Drive에 저장

In [ ]:
def fetch(name):
    return urllib.request.urlopen(GITHUB_BASE + name, timeout=60).read()

man_bytes = fetch(MANIFEST)
man = pd.read_csv(io.BytesIO(man_bytes))
rows = []
for f, size, md5 in zip(man["file"], man["bytes"], man["md5"]):
    data = fetch(f)
    ok = hashlib.md5(data).hexdigest() == md5 and len(data) == size
    if ok:
        with open(os.path.join(TARGET, f), "wb") as fh:
            fh.write(data)
    rows.append({"파일": f, "크기(byte)": len(data), "MD5 확인": "일치 → 저장" if ok else "⚠ 불일치(저장 안 함)"})
for f in EXTRA:
    with open(os.path.join(TARGET, f), "wb") as fh:
        fh.write(fetch(f))
with open(os.path.join(TARGET, MANIFEST), "wb") as fh:       # 실습 노트북이 이 파일로 폴더를 찾음
    fh.write(man_bytes)

# 오프라인·업로드용 묶음 (Drive에서 내려받아 USB/PC에 보관 가능)
zpath = os.path.join(TARGET, "week05_data.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in list(man["file"]) + EXTRA + [MANIFEST]:
        z.write(os.path.join(TARGET, f), f)
pd.DataFrame(rows)

## 3. 저장 결과 확인

In [ ]:
check = []
for f, md5, n in zip(man["file"], man["md5"], man["rows"]):
    p = os.path.join(TARGET, f)
    good = os.path.exists(p) and hashlib.md5(open(p, "rb").read()).hexdigest() == md5
    check.append({"파일": f, "행 수": n, "Drive 저장본": "정상" if good else "⚠ 확인 필요"})
print("폴더:", TARGET)
print("파일 목록:", sorted(os.listdir(TARGET)))
pd.DataFrame(check)

## 4. 학생에게 공유하는 방법
1. Drive에서 `02_데이터` 폴더 → **공유** → 수업 학생(또는 링크가 있는 사용자) **뷰어** 권한
2. 학생: 공유받은 폴더를 오른쪽 클릭 → **정리 → 바로가기 추가 → 내 드라이브**
3. 학생: 실습 노트북 0장에서 `DATA_SOURCE = "drive"` 로 바꾸고 실행 → 노트북이 `week05_MANIFEST.csv`가 있는 폴더를 자동으로 찾음
4. Drive 연결이 어려운 학생: `week05_data.zip`을 내려받아 압축을 풀고 `DATA_SOURCE = "upload"` → CSV와 `week05_MANIFEST.csv` 선택

> 바로가기를 추가하지 않으면 '공유 문서함'의 파일은 Colab의 `/content/drive/MyDrive`에 보이지 않습니다.

## (선택) 최신 날짜까지 데이터 갱신
수업 기준 데이터는 2026-10-13까지입니다. 더 최근 데이터가 필요하면 아래 셀의 `REFRESH = True`로 바꿔 실행합니다.
결과는 기존 파일을 덮어쓰지 않고 `latest_YYYYMMDD` 하위 폴더에 저장됩니다. **기준일이 바뀌면 체크리스트 기대값과 다른 값이 나옵니다.**

In [ ]:
REFRESH = False
END_DATE = pd.Timestamp.now(tz="Asia/Seoul").strftime("%Y-%m-%d")

if REFRESH:
    !git clone -q --depth 1 https://github.com/FinanceData/marcap.git /content/marcap
    !pip -q install finance-datareader pyarrow
    !wget -q -O /content/build_week05_data.py https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week05/tools/build_week05_data.py
    import sys, importlib.util, pathlib
    spec = importlib.util.spec_from_file_location("bw", "/content/build_week05_data.py")
    bw = importlib.util.module_from_spec(spec); spec.loader.exec_module(bw)
    out = pathlib.Path(TARGET) / ("latest_" + END_DATE.replace("-", ""))
    bw.OUT = out
    bw.main("/content/marcap", "2025-01-01", END_DATE)
    print("갱신 데이터 저장:", out, sorted(os.listdir(out)))